# FDSP Funcition

act like as DDP


```python

from torch.distributed.fsdp import FullySharedDistributedParallel as FSDP


model = nn.Sequential(
    nn.Linear(10, 10)
)

model = FSDP(model)

# 2. Arguments

## 2.1 Auto Wrap Policy

`auto_wrap_policy` defines **where** FSDP splits the model into separate units (each unit gets its own all-gather / re-shard cycle).

If `auto_wrap_policy=None`, the whole model becomes a single unit — meaning **all parameters get all-gathered at once**, before any computation starts. This defeats the purpose of FSDP, since you lose the memory savings and the overlap between communication and computation.

There are 2 main ways to define this policy:

### 2.1.1 `size_based_auto_wrap_policy`

Splits the model based on **parameter count**, not structure. It walks through the model, accumulating parameters module by module, until the accumulated total reaches `min_num_params`. When it does, it closes that group as one FSDP unit and resets the counter for the next group.

```python
policy = lambda module, recurse, nonwrapped_numel: size_based_auto_wrap_policy(
    module,
    recurse,
    nonwrapped_numel,
    min_num_params=1_500_000  # keeps accumulating modules until it reaches 1.5M params, then wraps them together as one unit
)

model = FSDP(
    auto_wrap_policy = policy
)


```

**Note:** `size_based_auto_wrap_policy` expects `(module, recurse, nonwrapped_numel)` — these 3 args are passed automatically by FSDP during traversal. Since it also needs a 4th arg (`min_num_params`), we wrap it in a `lambda` to "lock in" that value before FSDP calls it.


## 2.2 Sharding Strategy

`sharding_strategy` defines **what** gets sharded across GPUs — controls the trade-off between memory savings and communication overhead.

```python
from torch.distributed.fsdp import ShardingStrategy

model = FSDP(
    sharding_strategy=ShardingStrategy.FULL_SHARD,
)
```

Main options:

| Strategy | What it shards | Memory | Communication |
|---|---|---|---|
| `FULL_SHARD` | params + gradients + optimizer states | Lowest | Highest (all-gather on every forward/backward) |
| `SHARD_GRAD_OP` | gradients + optimizer states only (params stay replicated) | Medium | Medium (like DDP + ZeRO-2 style) |
| `NO_SHARD` | nothing (equivalent to plain DDP) | Highest | Lowest |
| `HYBRID_SHARD` | full shard within a node, replicate across nodes | Balanced for multi-node | Reduces cross-node traffic |

`FULL_SHARD` is the default choice for maximizing memory savings — usually the go-to when training large models that don't fit otherwise.

## 2.3 Sync Module States

`sync_module_states` ensures all ranks start with **identical weights**, by broadcasting the parameters from rank 0 to all other ranks during FSDP initialization.

```python
model = FSDP(
    sync_module_states=True
)
```

**Use `True` when:**
- Each rank loads the model independently (e.g. from disk) and there's a risk of small differences (different random init, partial checkpoint loading, etc.)
- You want a guarantee that rank 0's weights are the source of truth for everyone.

**Use `False` (default) when:**
- The model is already guaranteed identical across ranks before wrapping (e.g. same checkpoint loaded deterministically on all ranks).
- You want to skip the extra broadcast overhead at startup.

In [ ]:
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullySharedDistributedParallel as FSDP, ShardingStrategy
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
import os
import torch

# Create the connection between the GPUs
dist.init_process_group(backend='nccl')
local_rank = int(os.environ["LOCAL_RANK"])
world_size = int(os.environ["WORLD_SIZE"])
torch.cuda.set_device(local_rank)

# Create the Model
model = nn.Sequential(
    nn.Linear(10, 10).
    nn.Relu()
).cuda()

# Create the policy for the auto_wrap
policy = lambda m, n, r: size_based_auto_wrap_policy(
    m,
    n,
    r,
    min_num_params= 1500000 # 1.5M
)

# Do the FSDP
model = FSDP(
    module = model, 
    sharding_strategy = ShardingStrategy.FULL_SHARD, 
    device_id = local_rank,
    sync_module_states = True,
    auto_wrap_policy = policy
)

# 3. Data Function (same as DDP)

Now that each process have their GPU and the same model, we need to remember the main objective of the Data Parallel, that's train the same model in differents GPUs and get the medium gradient to update, but the data in each process need to be different cuz if it's the same will be useless, so we need to use an function to do each process have a different data. So we use the `DistributedSampler()`

And the arguments
- `dataset` = The dataset, needed to get acess of the indexs
- `num_replicas` = It's the world_size, will split in groups of indexs
- `rank` = It's the actual local_rank, need to get the group of the index

And we pass this sampler for the loader

```python

from torch.utils.data.distributed import DistributedSampler  # Import the sampler

sampler  = DistributedSampler(
    dataset = dataset,
    rank = local_rank,
    num_replicas = world_size,
    shuffle = True
)

loader = DataLoader(
    dataset = dataset,
    sampler = sampler,
    batch_size = 32 # THIS BATCH_SIZE WILL BE THE NUMBER FOR EACH GPU (IF 4 GPUS = 4 * 32)
    shuffle = False # If you use shuffle in the sampler you can't use in the loader
)
```

# 4. Checkpoint (FSDP checkpoint)

to add
